# Tutorial 12: Single-Cell Foundation Model Embeddings

This notebook demonstrates how to extract **cell embeddings** from
state-of-the-art single-cell RNA-seq foundation models using `embpy`.

All models are accessed through the
[Helical](https://github.com/helicalAI/helical) package, providing
a unified interface over:

| Family | Models available | Reference |
|---|---|---|
| **scGPT** | 33M-cell transformer | [Cui et al. 2024](https://doi.org/10.1038/s41592-024-02201-0) |
| **Geneformer** | v1 (6L/12L), v2 (12L/18L/20L), cancer-tuned | [Theodoris et al. 2023](https://doi.org/10.1038/s41586-023-06139-9) |
| **UCE** | Cross-species (36M cells) | [Rosen et al. 2023](https://doi.org/10.1101/2023.11.28.568918) |
| **TranscriptFormer** | Metazoa / Exemplar / Sapiens (CZI) | [Fang et al. 2025](https://doi.org/10.1101/2025.04.25.650731) |
| **Tahoe-x1** | 70M / 1B / 3B | — |
| **Cell2Sentence-Scale** | 2B (Pythia) / 27B (Gemma-2) | — |

### What you'll learn

1. List and inspect available single-cell foundation models
2. Load a model and extract cell embeddings from an AnnData object
3. Compare embeddings across different models
4. Visualise cell embedding spaces with PCA and UMAP
5. Store embeddings in AnnData for downstream analysis

### Requirements

```bash
pip install embpy[helical]
```

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

## 1. Discover available models

`embpy` provides a registry of all supported single-cell foundation models.

In [ ]:
from embpy.models.singlecell_models import (
    list_singlecell_models,
    singlecell_info,
    get_singlecell_wrapper,
)

models = list_singlecell_models()
print(f"Available single-cell models ({len(models)}):")
for m in models:
    card = singlecell_info(m)
    print(f"  {m:35s} — {card.description}")

## 2. Prepare example data

Single-cell models expect an `AnnData` with **raw counts**. We create a
small example here; in practice, use your own scRNA-seq dataset.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

np.random.seed(42)
n_cells, n_genes = 100, 2000

gene_names = [f"GENE_{i}" for i in range(n_genes)]
cell_ids = [f"cell_{i}" for i in range(n_cells)]
cell_types = np.random.choice(["T cell", "B cell", "Monocyte"], size=n_cells)

counts = np.random.poisson(lam=3, size=(n_cells, n_genes)).astype(np.float32)

adata = ad.AnnData(
    X=counts,
    obs=pd.DataFrame({"cell_type": cell_types}, index=cell_ids),
    var=pd.DataFrame(index=gene_names),
)
adata

## 3. Load a model and extract embeddings

The workflow is always three steps:

1. **Instantiate** a wrapper (via the factory or directly)
2. **Load** the model weights onto a device
3. **Embed** cells from an AnnData

### 3a. scGPT

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
scgpt = get_singlecell_wrapper("scgpt", batch_size=10)
scgpt.load(device)

embs_scgpt = scgpt.embed_cells(adata)
print(f"scGPT embeddings: {embs_scgpt.shape}")
print(f"First cell (first 10 dims): {embs_scgpt[0, :10]}")

### 3b. Geneformer

In [ ]:
gf = get_singlecell_wrapper("geneformer_v2_12L", batch_size=10)
gf.load(device)

embs_gf = gf.embed_cells(adata)
print(f"Geneformer embeddings: {embs_gf.shape}")

### 3c. UCE

In [ ]:
uce = get_singlecell_wrapper("uce", batch_size=10)
uce.load(device)

embs_uce = uce.embed_cells(adata)
print(f"UCE embeddings: {embs_uce.shape}")

### 3d. TranscriptFormer (CZI)

In [ ]:
tf = get_singlecell_wrapper("transcriptformer_sapiens", batch_size=10)
tf.load(device)

embs_tf = tf.embed_cells(adata)
print(f"TranscriptFormer-Sapiens embeddings: {embs_tf.shape}")

## 4. Compare embedding spaces

We can compare how different models organise cells by computing
pairwise cosine similarities or reducing to 2D.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

embedding_dict = {
    "scGPT": embs_scgpt,
    "Geneformer v2": embs_gf,
    "UCE": embs_uce,
    "TranscriptFormer": embs_tf,
}

fig, axes = plt.subplots(1, len(embedding_dict), figsize=(5 * len(embedding_dict), 5))

for ax, (name, embs) in zip(axes, embedding_dict.items()):
    pca = PCA(n_components=2)
    coords = pca.fit_transform(embs)
    for ct in np.unique(cell_types):
        mask = cell_types == ct
        ax.scatter(coords[mask, 0], coords[mask, 1], label=ct, alpha=0.6, s=15)
    ax.set_title(name)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Store embeddings in AnnData

It's convenient to store embeddings in `adata.obsm` for downstream analyses.

In [ ]:
adata.obsm["X_scgpt"] = embs_scgpt
adata.obsm["X_geneformer"] = embs_gf
adata.obsm["X_uce"] = embs_uce
adata.obsm["X_transcriptformer"] = embs_tf

print("Stored embedding keys:", list(adata.obsm.keys()))
adata

## 6. Using the wrapper classes directly

Instead of the factory, you can instantiate wrappers directly for
more control over configuration.

In [ ]:
from embpy.models.singlecell_models import ScGPTWrapper, GeneformerWrapper

wrapper = ScGPTWrapper(batch_size=16)
wrapper.load(device)
print(f"Loaded: {wrapper}")
print(f"Embedding dim: {wrapper.embedding_dim}")

## 7. Larger models: Tahoe-x1 and Cell2Sentence-Scale

These models are larger and may require more GPU memory.

In [ ]:
# Tahoe-x1 (70M — smallest variant)
tahoe = get_singlecell_wrapper("tahoe_70m", batch_size=10)
tahoe.load(device)

embs_tahoe = tahoe.embed_cells(adata)
print(f"Tahoe-x1 70M embeddings: {embs_tahoe.shape}")

In [ ]:
# Cell2Sentence-Scale (2B — Gemma-2-based)
c2s = get_singlecell_wrapper("cell2sentence_2b", batch_size=5)
c2s.load(device)

embs_c2s = c2s.embed_cells(adata)
print(f"Cell2Sentence-Scale 2B embeddings: {embs_c2s.shape}")

## 8. Cosine similarity between cell embeddings

Compute pairwise cosine similarity to check how well models
separate cell types.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_scgpt = cosine_similarity(embs_scgpt)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_scgpt, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("scGPT cell-cell cosine similarity")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Summary

This tutorial showed how to:

- **List** all 20+ single-cell foundation model variants via `list_singlecell_models()`
- **Load** any model with the 3-step pattern: instantiate → load → embed
- **Compare** embedding spaces across models using PCA visualisation
- **Store** embeddings in `adata.obsm` for seamless integration with scanpy/scvi

Key functions:

```python
from embpy.models.singlecell_models import (
    list_singlecell_models,   # discover available models
    singlecell_info,          # model metadata
    get_singlecell_wrapper,   # factory function
)
```

For the full list of models and variants, see the
[API reference](../api.md) or run `list_singlecell_models()`.